In [ ]:
import sys
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir      = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config
import triage_filter

cfg = load_config()

# Resolve triage path (new key — created by run_triage if absent)
triage_path = Path(cfg["paths"]["triage"])

print("Config loaded.")
print("  raw_images :", cfg["paths"]["raw_images"])
print("  triage     :", triage_path)


# Stage 00a.2 — SigLIP Triage Filter

**What this notebook does:**  
Scores every raw patent image with [SigLIP](https://huggingface.co/timm/ViT-SO400M-14-SigLIP-384) (a large vision-language model) to detect images that are **not technical drawings** — tables, text pages, data grids, title sheets — before those images enter the expensive labelling pipeline downstream.

It is **completely non-destructive**: nothing in `raw/` is touched. Outputs are JSON/CSV manifests only.

---

## Why this step exists

PatSeer downloads *everything* on the Drawings tab — which often includes:
- Cover pages / title sheets
- Tables of component labels
- Text-only description pages
- Flowcharts and block diagrams that look like drawings but aren't figure crops

Running `00b` (figure crop + label matching) on those wastes GPU time and pollutes the labelled dataset. This step flags them so Stage `00b` can skip them.

---

## How the scoring works

Each image is scored against two zero-shot text prompts using SigLIP:

| Prompt | Meaning |
|--------|---------|
| `"a technical patent line drawing of an aircraft or mechanical component"` | → `score_drawing` |
| `"a table, chart, text page, or data grid with no technical drawing"` | → `score_table` |

The two scores are softmax-normalised so they sum to 1.  
`keep = True` when `score_drawing ≥ threshold` (default **0.60**).  
`keep = False` = flagged as non-drawing → will be skipped by `00b`.

> **Threshold guide:**  
> `0.60` (default) — conservative, keeps borderline images, low false-positive rate  
> `0.65` — moderate filtering  
> `0.70` — aggressive, only keep high-confidence drawings

---

## Outputs

Both written under `cfg["paths"]["triage"]` = `1639_DS/triage/`:

| File | Content |
|------|---------|
| `<patent_id>.json` | Per-image scores for one patent: `file`, `pred`, `score_drawing`, `score_table`, `keep` |
| `triage_summary.csv` | One row per patent: `patent_id`, `total_images`, `flagged_count`, `flagged_ratio` |

Already-processed patents are **not re-scored** on repeat runs (JSON exists → skip).

---

## Where it fits in the pipeline

```
00a    (download from PatSeer)
 ↓
00a.1  (audit: which patents are missing / need re-download)
 ↓
00a.2  ← YOU ARE HERE  (score every image: drawing vs. non-drawing)
 ↓
00b    (figure crop + _F/_Fu label matching — uses triage to skip flagged images)
 ↓
02     (pad + resize to 518×518)
```

---

## Cell guide

| Cell | What it does |
|------|--------------|
| 1 | Load config — resolves `raw_images` and `triage` paths |
| 2 | Load SigLIP model (~3 GB weights, cached after first run, needs GPU) |
| 3 | Run triage — set `limit=10` to test on 10 patents first, `limit=None` for full dataset |
| 4 | Inspect results — flagged ratio summary + per-image table for worst patents |

---

## Before you run

- **GPU required** — SigLIP ViT-SO400M is large; CPU inference is very slow (~10 s/image).  
- **First run downloads ~3 GB** of weights into the HuggingFace cache (`~/.cache/huggingface/`).  
- Run Cell 3 with `limit=10` first to check the threshold is sensible for your data before processing all 1350 folders.  
- If scores cluster around 0.50–0.52 (as in the test output above), the prompts may need tuning for your specific patent images — see `src/triage_filter.py` → `_PROMPT_KEEP` / `_PROMPT_FLAG`.

In [ ]:
# Loads SigLIP ViT-SO400M-14-SigLIP-384 via open_clip.
# Downloads ~3 GB of weights on first run (cached by HuggingFace Hub).
model, processor, device = triage_filter.load_siglip_model(cfg)
print(f"SigLIP ready on: {device}")


In [ ]:
# ── Run options ──────────────────────────────────────────────────────────────
#
# This cell runs SigLIP scoring — slow, and only needed ONCE per patent (or
# when new patent folders are added). Already-triaged patents are skipped by
# default so locked review decisions are never touched.
#
# To just change the keep/discard threshold on already-scored images, do NOT
# run this cell — use Cell 3b below instead (rethreshold_existing / reset_threshold),
# which is instant and never re-runs the model.
#
TEST_MODE  = False   # True = only 10 patents (quick sanity check)
                    # False = full dataset (all ~1614 folders)
THRESHOLD  = 0.53   # kept for API compatibility; logic now uses pure argmax
                    # (discard only when model prefers the table/text prompt)
FORCE      = False  # True = re-score and OVERWRITE already-triaged patents,
                    # destroying any locked review decisions. Only use this
                    # intentionally (e.g. raw images changed).
# ─────────────────────────────────────────────────────────────────────────────

limit = 600 if TEST_MODE else None
print(f"Running triage: {'TEST MODE (10 patents)' if TEST_MODE else 'FULL DATASET'}, threshold={THRESHOLD}")
triage_filter.run_triage(cfg, threshold=THRESHOLD, limit=limit, force=FORCE)

## Cell 3b — Re-threshold existing results (no GPU needed)

Already-confirmed discards (`keep=False`) are **never** upgraded back.  
Only images currently `keep=True` are re-evaluated against the new threshold.  
Rewrites all JSONs and `triage_summary.csv` in-place.

In [ ]:
import importlib, triage_filter
importlib.reload(triage_filter)   # pick up src edits without restarting kernel

# ── Choose mode ───────────────────────────────────────────────────────────────
#
#   "add"   — rethreshold_existing: only ADDS new discards, never re-enables
#             anything already marked keep=False (safe, one-way)
#
#   "reset" — reset_threshold: re-derives keep from raw scores for ALL images,
#             works in both directions — use this to undo an over-aggressive run
#
MODE      = "reset"   # "add" | "reset"
THRESHOLD = 0.51      # tune this valuel
# ─────────────────────────────────────────────────────────────────────────────

if MODE == "add":
    triage_filter.rethreshold_existing(cfg, new_threshold=THRESHOLD)
elif MODE == "reset":
    triage_filter.reset_threshold(cfg, threshold=THRESHOLD)
else:
    print(f"Unknown MODE={MODE!r} — choose 'add' or 'reset'")


## Cell 5b — Lock/unlock (advanced)

Use **only** if you need direct control over locks outside the normal review flow.

| Action | Effect |
|--------|--------|
| `lock_discards` | Lock all current DISCARDs — they won't reappear in the viewer |
| `lock_keeps` | Lock all current KEEPs — aggressive threshold won't remove them |
| `unlock_all` | Clear every lock (full clean slate) |
| `stats` | Show current lock counts (no writes) |

> **Normal workflow:** use the Cell 6 viewer + the Commit cell at the bottom instead.

In [ ]:
import importlib, triage_filter
importlib.reload(triage_filter)

# ── Choose lock action ────────────────────────────────────────────────────────
#
#   "lock_discards"  — protect current DISCARDs; threshold raises cannot re-enable them
#   "lock_keeps"     — protect current KEEPs; threshold raises cannot discard them
#   "unlock_all"     — clear all locks (clean slate before a full reset)
#   "stats"          — just print the current lock counts (no writes)
#   None             — do nothing
#
LOCK_ACTION = "stats"   # change to "lock_discards", "lock_keeps", or "unlock_all"
# ─────────────────────────────────────────────────────────────────────────────

if LOCK_ACTION == "lock_discards":
    triage_filter.lock_discards(cfg)
elif LOCK_ACTION == "lock_keeps":
    triage_filter.lock_keeps(cfg)
elif LOCK_ACTION == "unlock_all":
    triage_filter.unlock_all(cfg)
elif LOCK_ACTION == "stats":
    triage_filter.locked_stats(cfg)
else:
    print("LOCK_ACTION not set — nothing done.")


In [ ]:
import json
import pandas as pd

triage_path = Path(cfg["paths"]["triage"])
summary_csv = triage_path / "triage_summary.csv"

if not summary_csv.exists():
    print("No triage_summary.csv found — run Cell 3 first.")
else:
    summary_df = pd.read_csv(summary_csv)

    total_images  = summary_df["total_images"].sum()
    total_flagged = summary_df["flagged_count"].sum()
    overall_ratio = total_flagged / max(1, total_images)

    print(f"Total images scored : {total_images}")
    print(f"Total flagged       : {total_flagged}")
    print(f"Overall flagged ratio: {overall_ratio:.1%}")
    print()

    # Show per-figure results for the 3 patents with the most flagged images
    top3 = summary_df.nlargest(3, "flagged_count")[["patent_id", "total_images", "flagged_count", "flagged_ratio"]]
    print("Top 3 patents by flagged count:")
    display(top3.reset_index(drop=True))
    print()

    for patent_id in top3["patent_id"]:
        json_path = triage_path / f"{patent_id}.json"
        if not json_path.exists():
            print(f"  {patent_id}: JSON not found")
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        figures_df = pd.DataFrame(data["figures"])
        print(f"\n--- {patent_id}  (total={data['total']}  flagged={data['flagged']}) ---")
        display(figures_df[["file", "pred", "score_drawing", "score_table", "keep"]])


## Cell 5 — Visual inspection: kept vs. discarded images

Shows the actual images for one patent so you can judge whether the threshold is working correctly.  
- **Green border** = kept (`keep=True`, `score_drawing ≥ threshold`)  
- **Red border** = discarded (`keep=False`)  

Change `INSPECT_PATENT` to any patent ID from the triage results, or leave it as `None` to auto-pick the first processed patent.

In [ ]:
import json
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image

triage_path = Path(cfg["paths"]["triage"])
raw_dir     = Path(cfg["paths"]["raw_images"])
MAX_COLS    = 4

# All processed patents, sorted
patent_ids = sorted(p.stem for p in triage_path.glob("*.json")
                    if p.stem != "triage_summary")
if not patent_ids:
    print("No triage JSON files found — run Cell 3 first.")
    raise SystemExit

print(f"Found {len(patent_ids)} processed patents. Use the buttons or dropdown to navigate.")

# ── Widgets ───────────────────────────────────────────────────────────────────
idx_state   = {"i": 0, "updating": False}   # flag prevents dropdown re-entrancy
out         = widgets.Output()
btn_prev    = widgets.Button(description="◀ Prev", button_style="info",  layout=widgets.Layout(width="120px"))
btn_next    = widgets.Button(description="Next ▶", button_style="info",  layout=widgets.Layout(width="120px"))
lbl_pos     = widgets.Label(value=f"1 / {len(patent_ids)}")
patent_dd   = widgets.Dropdown(options=patent_ids, value=patent_ids[0],
                                layout=widgets.Layout(width="360px"))

def show_patent(patent_id):
    json_path = triage_path / f"{patent_id}.json"
    with open(json_path) as fh:
        data = json.load(fh)
    figures = data["figures"]
    n       = len(figures)
    if n == 0:
        print(f"{patent_id}: no figures in triage JSON")
        return
    n_kept  = sum(1 for f in figures if f["keep"])
    n_disc  = n - n_kept

    n_cols = min(MAX_COLS, n)
    n_rows = math.ceil(n / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 1.8, n_rows * 2.0),
                             squeeze=False, dpi=80)
    flat_axes = [ax for row in axes for ax in row]

    patent_dir = raw_dir / patent_id
    for ax, fig_info in zip(flat_axes, figures):
        img_path = patent_dir / fig_info["file"]
        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "not found", ha="center", va="center",
                    color="grey", transform=ax.transAxes)
        keep  = fig_info["keep"]
        color = "#2ecc71" if keep else "#e74c3c"
        label = "KEEP" if keep else "DISCARD"
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
        ax.set_title(f"{fig_info['file']}\n{label}  {fig_info['score_drawing']:.3f}",
                     fontsize=7, color=color, pad=3)
        ax.set_xticks([]); ax.set_yticks([])

    for ax in flat_axes[n:]:
        ax.axis("off")

    keep_patch    = mpatches.Patch(color="#2ecc71", label=f"Kept ({n_kept})")
    discard_patch = mpatches.Patch(color="#e74c3c", label=f"Discarded ({n_disc})")
    fig.legend(handles=[keep_patch, discard_patch], loc="lower center",
               ncol=2, fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"{patent_id}  —  {n_kept}/{n} kept  (threshold={THRESHOLD})",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

def refresh():
    i = idx_state["i"]
    # Update label and dropdown without triggering the observer callback
    lbl_pos.value = f"{i+1} / {len(patent_ids)}"
    idx_state["updating"] = True
    patent_dd.value = patent_ids[i]
    idx_state["updating"] = False
    with out:
        clear_output(wait=True)
        show_patent(patent_ids[i])

def on_prev(_):
    if idx_state["i"] > 0:
        idx_state["i"] -= 1
        refresh()

def on_next(_):
    if idx_state["i"] < len(patent_ids) - 1:
        idx_state["i"] += 1
        refresh()

def on_dropdown(change):
    # Only react to user-driven value changes, not programmatic ones from refresh()
    if change["name"] == "value" and not idx_state["updating"]:
        new_val = change["new"]
        if new_val in patent_ids:
            idx_state["i"] = patent_ids.index(new_val)
            refresh()

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
patent_dd.observe(on_dropdown)

nav_bar = widgets.HBox([btn_prev, lbl_pos, btn_next, patent_dd])
display(nav_bar, out)
refresh()

## Cell 6 — Review pending discards

Shows every image flagged as **discarded** that has **not yet been locked** (i.e. not reviewed in a previous session).

- **Red border** = DISCARD — will be permanently locked on Commit
- **Green border** = APPROVED — will be flipped to KEEP on Commit

Click any image to toggle between DISCARD and APPROVED.  Nothing is written to disk until you run the **Commit cell** below.

> Already-locked images (from previous sessions) are hidden — run `reset_review` in the Commit cell to bring them back.

In [ ]:
import importlib, triage_filter
importlib.reload(triage_filter)

import json, math, threading
import matplotlib
matplotlib.use("widget")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image

triage_path = Path(cfg["paths"]["triage"])
raw_dir     = Path(cfg["paths"]["raw_images"])

PAGE_SIZE = 200
MAX_COLS  = 5

# ── Load non-locked discarded images only ─────────────────────────────────────
# Images already locked (reviewed in a previous session) are hidden.
def load_pending_discards():
    result = []
    for json_path in sorted(triage_path.glob("*.json")):
        if json_path.stem == "triage_summary":
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        for fig in data["figures"]:
            if not fig["keep"] and not fig.get("locked"):
                result.append({
                    "patent_id":     json_path.stem,
                    "file":          fig["file"],
                    "score_drawing": fig["score_drawing"],
                    "score_table":   fig["score_table"],
                })
    return result

discarded = load_pending_discards()

# approved: images the user clicked to flip to KEEP this session (not yet on disk)
approved  = set()

# visited_pages: page indices actually rendered (shown) this session. Commit
# only locks images belonging to these pages — pages you never scroll to stay
# pending instead of being silently locked as DISCARD by default.
visited_pages = set()

n_total = len(discarded)
n_pages = max(1, math.ceil(n_total / PAGE_SIZE))
print(f"Pending review (non-locked discards): {n_total}  ({n_pages} pages of {PAGE_SIZE})")
print("  Red border  = DISCARD  (will be locked on Commit)")
print("  Green border = APPROVED (will be set to KEEP on Commit)")
print("  Click any image to toggle it to APPROVED.")
print("  Only pages you actually view here get committed — unvisited pages stay pending.")
print()
print("When done reviewing, run the LAST CELL to commit your decisions to disk.")

# ── Widgets ───────────────────────────────────────────────────────────────────
page_state = {"p": 0, "fig": None, "flat_axes": [], "batch": []}
out_disc   = widgets.Output()
btn_dp     = widgets.Button(description="◀ Prev", button_style="warning", layout=widgets.Layout(width="120px"))
btn_dn     = widgets.Button(description="Next ▶", button_style="warning", layout=widgets.Layout(width="120px"))
lbl_dp     = widgets.Label(value=f"Page 1 / {n_pages}")
lbl_info   = widgets.Label(value="")
lbl_counts = widgets.Label(value=f"Approved: 0 / {n_total}")
btn_commit = widgets.Button(description="Commit Page & Next", button_style="success", layout=widgets.Layout(width="180px"))
committed_pages = set()

def _ax_color(entry):
    key = (entry["patent_id"], entry["file"])
    return ("#2ecc71", "APPROVED ✓") if key in approved else ("#e74c3c", "DISCARD")

def show_page(p):
    batch  = discarded[p * PAGE_SIZE : (p + 1) * PAGE_SIZE]
    n      = len(batch)
    if n == 0:
        return
    visited_pages.add(p)
    n_cols = min(MAX_COLS, n)
    n_rows = math.ceil(n / max(n_cols, 1))

    if page_state["fig"] is not None:
        plt.close(page_state["fig"])

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 3.2, n_rows * 3.4),
                             squeeze=False, dpi=80)
    flat_axes = [ax for row in axes for ax in row]
    page_state.update({"fig": fig, "flat_axes": flat_axes, "batch": batch})

    for ax, entry in zip(flat_axes, batch):
        img_path = raw_dir / entry["patent_id"] / entry["file"]
        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "not found", ha="center", va="center",
                    color="grey", transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        color, label = _ax_color(entry)
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
        ax.set_title(
            f"{entry['patent_id']}\n{entry['file']}\n{label}  draw={entry['score_drawing']:.3f}",
            fontsize=6, color=color, pad=3
        )

    for ax in flat_axes[n:]:
        ax.axis("off")

    fig.suptitle(
        f"Pending discards — page {p+1}/{n_pages}  "
        f"(images {p*PAGE_SIZE+1}–{p*PAGE_SIZE+n} of {n_total})  "
        f"| Click image → APPROVED",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()

    def on_click(event):
        if event.inaxes is None:
            return
        fa, ba = page_state["flat_axes"], page_state["batch"]
        for i, ax in enumerate(fa[:len(ba)]):
            if ax is event.inaxes:
                entry = ba[i]
                key   = (entry["patent_id"], entry["file"])
                # toggle
                if key in approved:
                    approved.discard(key)
                else:
                    approved.add(key)
                color, label = _ax_color(entry)
                for spine in ax.spines.values():
                    spine.set_edgecolor(color)
                    spine.set_linewidth(4)
                ax.set_title(
                    f"{entry['patent_id']}\n{entry['file']}\n{label}  draw={entry['score_drawing']:.3f}",
                    fontsize=6, color=color, pad=3
                )
                fig.canvas.draw_idle()
                lbl_info.value   = f"{'✓ APPROVED' if key in approved else '✗ back to DISCARD'}: {entry['patent_id']} / {entry['file']}"
                lbl_counts.value = f"Approved: {len(approved)} / {n_total}"
                return

    fig.canvas.mpl_connect("button_press_event", on_click)
    plt.show()

def refresh_disc(new_p):
    page_state["p"] = new_p
    lbl_dp.value    = f"Page {new_p + 1} / {n_pages}"
    lbl_info.value  = ""
    with out_disc:
        clear_output(wait=True)
        show_page(new_p)

def commit_current_page(_):
    p = page_state["p"]
    batch = page_state["batch"]
    if not batch:
        return
    stats = triage_filter.commit_batch(cfg, batch, approved)
    committed_pages.add(p)
    lbl_info.value = (
        f"✓ Page {p+1} committed to disk — "
        f"{stats['newly_kept']} kept, {stats['newly_locked_discard']} discarded."
    )
    if p < n_pages - 1:
        refresh_disc(p + 1)
    else:
        with out_disc:
            clear_output(wait=True)
            print("Last page — all reviewed pages committed.")

btn_commit.on_click(commit_current_page)
btn_dp.on_click(lambda _: refresh_disc(page_state["p"] - 1) if page_state["p"] > 0 else None)
btn_dn.on_click(lambda _: refresh_disc(page_state["p"] + 1) if page_state["p"] < n_pages - 1 else None)

display(widgets.HBox([btn_dp, lbl_dp, btn_dn, btn_commit, lbl_counts, lbl_info]), out_disc)
refresh_disc(0)


## Cell 7 — Commit review decisions

Writes your Cell-6 review session to disk.

| `COMMIT_ACTION` | Effect |
|-----------------|--------|
| `"commit"` | Lock every APPROVED image as KEEP, lock every remaining DISCARD as discarded. They won't appear in the viewer again. |
| `"reset_review"` | Remove all discard locks — images reappear in Cell 6 for another review pass. Locked KEEPs are preserved. |
| `"stats"` | Show current lock counts (no writes). |

> **Typical session:**  
> 1. Run Cell 6, flip the false-positives to APPROVED  
> 2. Run this cell with `COMMIT_ACTION = "commit"`  
> 3. Next time you open the notebook, only *new* unreviewed discards show up  

> **To start fresh** (e.g. after changing threshold):  
> Set `COMMIT_ACTION = "reset_review"`, then re-run Cell 3b with `MODE = "reset"`

In [ ]:
import importlib, triage_filter
importlib.reload(triage_filter)

# ── Choose action ─────────────────────────────────────────────────────────────
#
#   "commit"        — write this session's decisions to disk (only pages you
#                     actually viewed in Cell 6 above — see reviewed_keys below)
#   "reset_review"  — unlock discard locks so they reappear in the viewer
#   "stats"         — show lock counts only (no writes)
#
COMMIT_ACTION = "commit"
# ─────────────────────────────────────────────────────────────────────────────

if COMMIT_ACTION == "commit":
    # `approved` and `visited_pages` are built up in Cell 6 above.
    # Only images on pages you actually viewed get committed — anything you
    # never scrolled to stays pending instead of being silently locked as
    # discard. If Cell 6 wasn't run this session, both default to empty so
    # this commit is a safe no-op rather than mass-locking everything pending.
    try:
        _approved = approved
        _visited_pages = visited_pages
    except NameError:
        _approved = set()
        _visited_pages = set()
        print("⚠  Cell 6 was not run this session — nothing to commit.")

    _reviewed_keys = {
        (fig["patent_id"], fig["file"])
        for p in _visited_pages
        for fig in discarded[p * PAGE_SIZE : (p + 1) * PAGE_SIZE]
    }
    triage_filter.commit_review(cfg, _approved, reviewed_keys=_reviewed_keys)

elif COMMIT_ACTION == "reset_review":
    triage_filter.reset_review(cfg)

elif COMMIT_ACTION == "stats":
    triage_filter.locked_stats(cfg)

else:
    print(f"Unknown COMMIT_ACTION={COMMIT_ACTION!r} — choose commit / reset_review / stats")


## Cell 8 — Browse ALL discarded images (read-only)

Shows every image currently `keep=False` (the full discard set, across every patent) — not just unlocked/pending ones like Cell 6. Purely for inspection: no click-to-approve, nothing is written to disk. Paginated, 5 columns per page, 500 images per page.


In [ ]:
import json, math
from pathlib import Path
from PIL import Image
import base64
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

triage_path = Path(cfg["paths"]["triage"])
raw_dir     = Path(cfg["paths"]["raw_images"])

PAGE_SIZE  = 500
N_COLS     = 5
THUMB_MAX  = 220   # px, kept small so 500 base64 thumbnails per page stays fast

def load_all_discards():
    result = []
    for json_path in sorted(triage_path.glob("*.json")):
        if json_path.stem == "triage_summary":
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        for fig in data["figures"]:
            if not fig["keep"]:
                result.append({
                    "patent_id":     json_path.stem,
                    "file":          fig["file"],
                    "score_drawing": fig.get("score_drawing"),
                    "score_table":   fig.get("score_table"),
                    "locked":        bool(fig.get("locked")),
                })
    return result

_all_discards = load_all_discards()
_n_total = len(_all_discards)
_n_pages = max(1, math.ceil(_n_total / PAGE_SIZE))
print(f"Total discarded images: {_n_total}  ({_n_pages} pages of up to {PAGE_SIZE})")

def _thumb_b64(path: Path) -> str:
    try:
        img = Image.open(path).convert("RGB")
        img.thumbnail((THUMB_MAX, THUMB_MAX))
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=70)
        return base64.b64encode(buf.getvalue()).decode()
    except Exception:
        return ""

_out = widgets.Output()
_prev_btn = widgets.Button(description="\u25c0 Prev", layout=widgets.Layout(width="100px"))
_next_btn = widgets.Button(description="Next \u25b6", layout=widgets.Layout(width="100px"))
_lbl = widgets.HTML()
_state = {"p": 0}

def _render(p):
    _state["p"] = p
    batch = _all_discards[p * PAGE_SIZE : (p + 1) * PAGE_SIZE]
    _lbl.value = (f"<b>Page {p+1} / {_n_pages}</b> &nbsp; "
                  f"(images {p*PAGE_SIZE+1}\u2013{p*PAGE_SIZE+len(batch)} of {_n_total})")
    cards = []
    for entry in batch:
        img_path = raw_dir / entry["patent_id"] / entry["file"]
        b64 = _thumb_b64(img_path)
        if b64:
            img_tag = (f'<img src="data:image/jpeg;base64,{b64}" '
                       f'style="max-width:100%;max-height:200px;object-fit:contain;"/>')
        else:
            img_tag = '<div style="color:grey;font-size:11px;">not found</div>'
        card = (
            '<div style="border:2px solid #e74c3c;border-radius:6px;padding:4px;'
            'display:flex;flex-direction:column;align-items:center;gap:2px;">'
            + img_tag +
            f'<div style="font-size:9px;color:#333;text-align:center;word-break:break-all;">'
            f'{entry["patent_id"]}<br>{entry["file"]}</div>'
            f'<div style="font-size:9px;color:#e74c3c;">'
            f'draw={entry["score_drawing"]:.3f}  table={entry["score_table"]:.3f}</div>'
            '</div>'
        )
        cards.append(card)
    html = (f'<div style="display:grid;grid-template-columns:repeat({N_COLS},1fr);gap:8px;">'
            + "".join(cards) + "</div>")
    with _out:
        clear_output(wait=True)
        display(HTML(html))

def _on_prev(_):
    if _state["p"] > 0:
        _render(_state["p"] - 1)

def _on_next(_):
    if _state["p"] < _n_pages - 1:
        _render(_state["p"] + 1)

_prev_btn.on_click(_on_prev)
_next_btn.on_click(_on_next)

display(widgets.HBox([_prev_btn, _lbl, _next_btn]), _out)
_render(0)


## Cell 9 — Rescue images from the discard pool

Click any image to toggle it **APPROVED** (green border) — click again to undo. Nothing is written to disk until you click **Commit Page**. Unlike Cell 6, this works on the full discard pool (already-locked images included) — rescued images are flipped to `keep=True` and stay locked (so a future threshold change can't silently discard them again).

Smaller page size than the Cell 8 browser (100 vs 500) since clicking needs a live matplotlib figure per image, not just a static HTML grid — 500 clickable subplots per page would be very slow to render.


In [ ]:
import importlib, triage_filter
importlib.reload(triage_filter)

import json, math
import matplotlib
matplotlib.use("widget")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image

triage_path = Path(cfg["paths"]["triage"])
raw_dir     = Path(cfg["paths"]["raw_images"])

RESCUE_PAGE_SIZE = 100
MAX_COLS         = 5

def load_all_discards_for_rescue():
    result = []
    for json_path in sorted(triage_path.glob("*.json")):
        if json_path.stem == "triage_summary":
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        for fig in data["figures"]:
            if not fig["keep"]:
                result.append({
                    "patent_id":     json_path.stem,
                    "file":          fig["file"],
                    "score_drawing": fig.get("score_drawing", 0.0),
                    "score_table":   fig.get("score_table", 0.0),
                })
    return result

_rescue_pool = load_all_discards_for_rescue()

# Thumbnail cache: decoding+downscaling each full-res scan is the slow part of
# rendering a page (patent sheets can be 1700x2200px+); cache the small,
# already-downscaled array so revisiting a page (e.g. clicking back to it) is
# instant instead of re-decoding 100 images from disk again.
_THUMB_MAX_PX = 200
_thumb_cache: dict = {}

def _get_thumb(img_path: Path):
    key = str(img_path)
    if key in _thumb_cache:
        return _thumb_cache[key]
    try:
        img = Image.open(img_path)
        img.draft("RGB", (_THUMB_MAX_PX, _THUMB_MAX_PX))   # fast partial JPEG/PNG decode at lower res when supported
        img = img.convert("RGB")
        img.thumbnail((_THUMB_MAX_PX, _THUMB_MAX_PX))
        arr = img
    except Exception:
        arr = None
    _thumb_cache[key] = arr
    return arr

# approved: images clicked this session, not yet written to disk
rescue_approved = set()
# rescued_committed: keys already written to disk this session (so re-visiting
# a page after commit shows them correctly as already-rescued, not re-offered)
rescued_committed = set()

n_total = len(_rescue_pool)
n_pages = max(1, math.ceil(n_total / RESCUE_PAGE_SIZE))
print(f"Discard pool: {n_total} images  ({n_pages} pages of up to {RESCUE_PAGE_SIZE})")
print("  Red border   = DISCARD (stays discarded)")
print("  Green border = APPROVED (will be rescued to KEEP on Commit)")
print("  Click any image to toggle. Click 'Commit Page' to write this page's decisions to disk.")

_rstate = {"p": 0, "fig": None, "flat_axes": [], "batch": []}
_rout      = widgets.Output()
_rprev_btn = widgets.Button(description="\u25c0 Prev", button_style="warning", layout=widgets.Layout(width="120px"))
_rnext_btn = widgets.Button(description="Next \u25b6", button_style="warning", layout=widgets.Layout(width="120px"))
_rpage_lbl = widgets.Label(value=f"Page 1 / {n_pages}")
_rinfo_lbl = widgets.Label(value="")
_rcount_lbl = widgets.Label(value=f"Approved: 0")
_rcommit_btn = widgets.Button(description="Commit Page", button_style="success", layout=widgets.Layout(width="140px"))

def _rcolor(entry):
    key = (entry["patent_id"], entry["file"])
    if key in rescued_committed:
        return ("#3498db", "RESCUED \u2713\u2713")
    if key in rescue_approved:
        return ("#2ecc71", "APPROVED \u2713")
    return ("#e74c3c", "DISCARD")

def show_rescue_page(p):
    batch = _rescue_pool[p * RESCUE_PAGE_SIZE : (p + 1) * RESCUE_PAGE_SIZE]
    n = len(batch)
    if n == 0:
        return
    n_cols = min(MAX_COLS, n)
    n_rows = math.ceil(n / max(n_cols, 1))

    if _rstate["fig"] is not None:
        plt.close(_rstate["fig"])

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.0, n_rows * 3.2), squeeze=False, dpi=70)
    flat_axes = [ax for row in axes for ax in row]
    _rstate.update({"fig": fig, "flat_axes": flat_axes, "batch": batch})

    for ax, entry in zip(flat_axes, batch):
        img_path = raw_dir / entry["patent_id"] / entry["file"]
        thumb = _get_thumb(img_path)
        if thumb is not None:
            ax.imshow(thumb)
        else:
            ax.text(0.5, 0.5, "not found", ha="center", va="center", color="grey", transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        color, label = _rcolor(entry)
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
        ax.set_title(f"{entry['patent_id']}\n{entry['file']}\n{label}  draw={entry['score_drawing']:.3f}",
                     fontsize=6, color=color, pad=3)

    for ax in flat_axes[n:]:
        ax.axis("off")

    fig.suptitle(f"Discard pool \u2014 page {p+1}/{n_pages}  (images {p*RESCUE_PAGE_SIZE+1}\u2013{p*RESCUE_PAGE_SIZE+n} of {n_total})  | Click \u2192 APPROVED",
                 fontsize=10, fontweight="bold")
    # tight_layout() does an expensive renderer pass over every subplot -- with
    # 100 of them per page that's a big chunk of render time. Fixed spacing
    # looks fine here since every subplot is the same size anyway.
    fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.01, wspace=0.15, hspace=0.45)

    def on_click(event):
        if event.inaxes is None:
            return
        fa, ba = _rstate["flat_axes"], _rstate["batch"]
        for i, ax in enumerate(fa[:len(ba)]):
            if ax is event.inaxes:
                entry = ba[i]
                key = (entry["patent_id"], entry["file"])
                if key in rescued_committed:
                    return  # already written to disk, nothing to toggle
                if key in rescue_approved:
                    rescue_approved.discard(key)
                else:
                    rescue_approved.add(key)
                color, label = _rcolor(entry)
                for spine in ax.spines.values():
                    spine.set_edgecolor(color)
                    spine.set_linewidth(4)
                ax.set_title(f"{entry['patent_id']}\n{entry['file']}\n{label}  draw={entry['score_drawing']:.3f}",
                             fontsize=6, color=color, pad=3)
                fig.canvas.draw_idle()
                _status_txt = "✓ APPROVED" if key in rescue_approved else "✗ back to DISCARD"
                _rinfo_lbl.value = f"{_status_txt}: {entry['patent_id']} / {entry['file']}"
                _rcount_lbl.value = f"Approved: {len(rescue_approved)}"
                return

    fig.canvas.mpl_connect("button_press_event", on_click)
    plt.show()

def _rrefresh(new_p):
    _rstate["p"] = new_p
    _rpage_lbl.value = f"Page {new_p + 1} / {n_pages}"
    _rinfo_lbl.value = ""
    with _rout:
        clear_output(wait=True)
        show_rescue_page(new_p)

def _rcommit(_):
    keys_this_page = {(e["patent_id"], e["file"]) for e in _rstate["batch"]}
    to_rescue = rescue_approved & keys_this_page
    if not to_rescue:
        _rinfo_lbl.value = "Nothing approved on this page."
        return
    stats = triage_filter.rescue_discards(cfg, to_rescue)
    rescued_committed.update(to_rescue)
    rescue_approved.difference_update(to_rescue)
    _rinfo_lbl.value = f"\u2713 Committed \u2014 {stats['rescued']} image(s) rescued to KEEP."
    # Must re-render INSIDE _rout (same as _rrefresh does for Prev/Next) -- calling
    # show_rescue_page() directly here renders a second, stray, live matplotlib
    # figure straight into the cell's raw output instead of replacing the one
    # inside the Output widget, leaving two interactive figures on screen at once
    # and making subsequent Prev/Next clicks target the wrong (stale) one.
    with _rout:
        clear_output(wait=True)
        show_rescue_page(_rstate["p"])

_rcommit_btn.on_click(_rcommit)
_rprev_btn.on_click(lambda _: _rrefresh(_rstate["p"] - 1) if _rstate["p"] > 0 else None)
_rnext_btn.on_click(lambda _: _rrefresh(_rstate["p"] + 1) if _rstate["p"] < n_pages - 1 else None)

display(widgets.HBox([_rprev_btn, _rpage_lbl, _rnext_btn, _rcommit_btn, _rcount_lbl, _rinfo_lbl]), _rout)
_rrefresh(0)
